# MRMS Data

In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

In [ ]:
from pansat.products.ground_based import stage4
from pansat.time import TimeRange

In [ ]:
from pansat.utils import make_latlon_area
lon_min = -125
lat_min = 24
lon_max = -65
lat_max = 51
imerg_grid = make_latlon_area(
    lon_min,
    lat_min,
    lon_max,
    lat_max,
    (lon_max - lon_min) / 0.1, 
    (lat_max - lat_min) / 0.1
)
imerg_grid

In [ ]:
from datetime import datetime
from pansat.utils import resample_data_binned
from pansat.products.ground_based import mrms

def extract_mrms_data(year, month, day, hour) -> xr.Dataset:
    """
    Download MRMS data and map to CONUS domain.

    Args:
        year: The year
        month: the month
        day: the day
        hout the hour

    Return:
        The MRMS multi sensor precip rate and the radar quality index half an hour after the full hour.
    """
    time = datetime(year, month, day, hour, minute=30)
    data_precip = mrms.precip_1h_ms.get(time)[0].open().rename(precip_1h_ms="surface_precip")
    time = data_precip.time.data
    data_rqi = mrms.radar_quality_index.get(time)[0].open().drop_vars(["time"])
    data = resample_data_binned(xr.merge([data_precip, data_rqi]), imerg_grid)
    data["time"] = time
    return data
    

In [ ]:
output_path = Path("/gdata2/simon/gprof_ir/conus/mrms")
output_path.mkdir(parents=True, exist_ok=True)

In [ ]:
start_time = np.datetime64("2022-08-01")
end_time = np.datetime64("2023-01-01")


for hour in np.arange(start_time, end_time, np.timedelta64(1, "h")):
    date = hour.astype("datetime64[s]").item()
    print(date)
    output_file = output_path / date.strftime("mrms_%Y%m%d%H%M%S.nc")
    if not output_file.exists():
        try:
            mrms_data = extract_mrms_data(date.year, date.month, date.day, date.hour)
            for var in ["surface_precip", "radar_quality_index"]:
                mrms_data[var].encoding = {
                    "zlib": True,
                    "dtype": "float32"
                }
            mrms_data.to_netcdf(output_file)
        except Exception:
            print("Error processing date: ", date)
    